In [1]:
# pip install qdrant-client
# pip install sentence-transformers

import os

import numpy as np
import pandas as pd

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

In [2]:
client_qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)

In [3]:
client_openai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [4]:
index_name = "ppgegc_qa_method"
dimensions = 1536

In [5]:
def histogram(classes:list, k:int):
    result = {}
    ctr=1
    for value, key in sorted(((classes.count(e), e) for e in set(classes)), reverse=True):
        if (ctr > k): break
        result[key] = value
        ctr+=1
    return result

In [6]:
def process_result(accuracy_dict, k, n, type):
    index = "{}-{}-{}".format(k,n,type)
    if (index in accuracy_dict):
        accuracy_dict[index] = accuracy_dict.get(index) + 1
    else:
        accuracy_dict[index] = 1

In [7]:
def get_process_result(accuracy_dict, k, n, type):
    index = "{}-{}-{}".format(k,n,type)
    if (index in accuracy_dict):
        return accuracy_dict[index]
    else:
        return 0

In [8]:
def print_process_result(accuracy_dict, k_list, n_list):
    for k in k_list:
        for n in n_list:
            positive = get_process_result(accuracy_dict, k, n, 'positive')
            negative = get_process_result(accuracy_dict, k, n, 'negative')
            accuracy = positive / (positive + negative)
            print("k={} - n={} - Positive: {} - Negative: {} - "
                "Accuracy: {} ".format(k,n,positive,negative,accuracy))

In [9]:
def transform_process_result(accuracy_dict, k_list, n_list):
    matrix = np.zeros((len(k_list), len(n_list)))
    i = j = 0
    for k in k_list:
        j=0
        for n in n_list:
            positive = get_process_result(accuracy_dict, k, n, 'positive')
            negative = get_process_result(accuracy_dict, k, n, 'negative')
            accuracy = positive / (positive + negative)
            matrix[i][j] = accuracy
            j+=1
        i+=1
    return matrix

In [10]:
#Method to approximate search
def search(vector):
    hits = client_qdrant.search(
        collection_name=index_name,
        query_vector=vector,
        limit=105 # 20, 50 e 100
    )
    return hits

In [13]:
PATH = "/home/thehprogrammer/Projects/python/qa-method/data/data.csv"

In [14]:
# Abrir csv em um dataframe
df = pd.read_csv(PATH)
df

,modalidade,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,palavras_chave,abstract,keywords,introducao,conclusao,embedding,x,y,z
0,treino,Jorge Ivan Hmeljevski,Modelo para Sistemas de Supervisão de Mercado ...,tese,Engenharia do Conhecimento,2021,Florianópolis,"Prof. José Leomar Todesco, Dr.","Prof. Alexandre Leopoldo Gonçalves, Dr.",A confiança na higidez dos mercados de capitai...,"['Mercado de Capitais', 'Mercado de Valores Mo...",The confidence in the integrity of capital mar...,"['Capital market', 'Securities Market', 'Marke...",{'contextualizacao': 'O mercado de capitais é ...,O modelo elaborado nest a pesquisa envolveu a ...,"[0.02262425608932972, 0.06360490620136261, 0.0...",-0.174124,0.159395,-0.060768
1,treino,IVAM GALVÃO FILHO,FRACTUS: APLICATIVO PARA APRENDIZAGEM DE FRAÇÕES,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Profª. Vania Ribas Ulbricht, Drª.","Profª. Elisa Maria Pivetta, Drª.",O objetivo principal desta pesquisa foi o dese...,"['Objetos de Aprendizagem.', 'Frações.', 'Apli...",The main objective of this research was the de...,"['Learning Objects.', 'Fractions.', 'App for L...",{'contextualizacao': 'O sistema educacional br...,O trabalho realizado pela organização TODOS PE...,"[-0.006245349999517202, 0.065089151263237, 0.0...",0.143998,0.263873,-0.055491
2,treino,MÁRCIO CRESCENCIO,MODELO DE UMA REDE COLABORATIVA SUPORTADA POR ...,tese,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Alexandre Augusto Biz , Dr.","Prof. José Leomar Todesco , Dr.",A convergência entre o turismo e a cultura atr...,"['Sítios de Patrimônio Mundial', 'Gestão do tu...",The convergence between tourism and culture th...,"['World Heritage Sites', 'Tourism management',...",{'contextualizacao': 'O turismo se tornou uma ...,Esta tese identificou que o turismo possui um ...,"[0.024945586919784546, 0.04501194879412651, 0....",-0.111669,-0.167909,-0.015695
3,treino,Roseli Honorio,MODELO CONCEITUAL DE GOVERNANÇA DE DADOS COMO ...,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Prof. João Artur de Souza, Dr.","Profa. Patrícia de Sá Freire, Dra.",A humanidade passou por transformações e revol...,"['Governança de Dados', 'Governança do Conheci...",Humanity has undergone transformations and rev...,"['Data Governance', 'Knowledge Governance', 'F...",{'contextualizacao': 'A sociedade está atraves...,"Na Ind ústria 4.0 e Sociedade 5.0, as organiza...","[0.05032069981098175, 0.051356106996536255, 0....",-0.229669,0.045873,-0.059402
4,treino,José Tadeu Silva,Análise da contribuição da engenharia do conhe...,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Fernando A. Ostuni Gauthier, Dr.","Prof. Marcelo Macedo, Dr.",A presente dissertação aborda as questões emer...,"['comércio eletrônico', 'modelo de análise', '...",The present dissertation addresses the emergin...,"['e-commerce', 'analysis model', 'knowledge en...",{'contextualizacao': 'De acordo com Lemos (200...,A partir do objetivo geral de analisar as cont...,"[-0.0012473083334043622, 0.044789645820856094,...",-0.195246,0.141014,0.098894
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,treino,Rosani Cesário Pereira,Competências essenciais dos pregoeiros: um est...,dissertação,Gestão do Conhecimento,2023,Florianópolis,"Prof.a Édis Mafra Lapolli, Dr a.","Prof.a Gertrudes A. Dandolini , Dra.",O modelo de gestão que vem se consolidando na ...,"['competências', 'competências essenciais', 'p...",The management model that has been consolidati...,"['competencies', 'essential competencies', 'pu...","{'contextualizacao': 'A autora, Rosani Cesário...",A pesquisa conclui que as competências essenci...,"[0.03604484349489212, 0.046274762600660324, 0....",-0.049497,-0.208519,-0.222047
146,treino,Fabíola Provensi,Gestão do Conhecimento e Cultura de Segurança ...,dissertação,Gestão do Conhecimento,2023,Florianópolis,"Prof. Eduardo Juan Soriano Sierra, Dr.","Prof. Neri Dos Santos, Dr.","

In [15]:
# Contagem deve se tornar o id
df["id"] = df.index

# Id como index
df.set_index("id", inplace=True)

df.head()

,modalidade,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,palavras_chave,abstract,keywords,introducao,conclusao,embedding,x,y,z
id,,,,,,,,,,,,,,,,,,,
0,treino,Jorge Ivan Hmeljevski,Modelo para Sistemas de Supervisão de Mercado ...,tese,Engenharia do Conhecimento,2021,Florianópolis,"Prof. José Leomar Todesco, Dr.","Prof. Alexandre Leopoldo Gonçalves, Dr.",A confiança na higidez dos mercados de capitai...,"['Mercado de Capitais', 'Mercado de Valores Mo...",The confidence in the integrity of capital mar...,"['Capital market', 'Securities Market', 'Marke...",{'contextualizacao': 'O mercado de capitais é ...,O modelo elaborado nest a pesquisa envolveu a ...,"[0.02262425608932972, 0.06360490620136261, 0.0...",-0.174124,0.159395,-0.060768
1,treino,IVAM GALVÃO FILHO,FRACTUS: APLICATIVO PARA APRENDIZAGEM DE FRAÇÕES,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Profª. Vania Ribas Ulbricht, Drª.","Profª. Elisa Maria Pivetta, Drª.",O objetivo principal desta pesquisa foi o dese...,"['Objetos de Aprendizagem.', 'Frações.', 'Apli...",The main objective of this research was the de...,"['Learning Objects.', 'Fractions.', 'App for L...",{'contextualizacao': 'O sistema educacional br...,O trabalho realizado pela organização TODOS PE...,"[-0.006245349999517202, 0.065089151263237, 0.0...",0.143998,0.263873,-0.055491
2,treino,MÁRCIO CRESCENCIO,MODELO DE UMA REDE COLABORATIVA SUPORTADA POR ...,tese,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Alexandre Augusto Biz , Dr.","Prof. José Leomar Todesco , Dr.",A convergência entre o turismo e a cultura atr...,"['Sítios de Patrimônio Mundial', 'Gestão do tu...",The convergence between tourism and culture th...,"['World Heritage Sites', 'Tourism management',...",{'contextualizacao': 'O turismo se tornou uma ...,Esta tese identificou que o turismo possui um ...,"[0.024945586919784546, 0.04501194879412651, 0....",-0.111669,-0.167909,-0.015695
3,treino,Roseli Honorio,MODELO CONCEITUAL DE GOVERNANÇA DE DADOS COMO ...,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Prof. João Artur de Souza, Dr.","Profa. Patrícia de Sá Freire, Dra.",A humanidade passou por transformações e revol...,"['Governança de Dados', 'Governança do Conheci...",Humanity has undergone transformations and rev...,"['Data Governance', 'Knowledge Governance', 'F...",{'contextualizacao': 'A sociedade está atraves...,"Na Ind ústria 4.0 e Sociedade 5.0, as organiza...","[0.05032069981098175, 0.051356106996536255, 0....",-0.229669,0.045873,-0.059402
4,treino,José Tadeu Silva,Análise da contribuição da engenharia do conhe...,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Fernando A. Ostuni Gauthier, Dr.","Prof. Marcelo Macedo, Dr.",A presente dissertação aborda as questões emer...,"['comércio eletrônico', 'modelo de análise', '...",The present dissertation addresses the emergin...,"['e-commerce', 'analysis model', 'knowledge en...",{'contextualizacao': 'De acordo com Lemos (200...,A partir do objetivo geral de analisar as cont...,"[-0.0012473083334043622, 0.044789645820856094,...",-0.195246,0.141014,0.098894


In [16]:
import ast

df['introducao_dict'] = df['introducao'].apply(ast.literal_eval)

# Depois, crie as novas colunas extraindo os valores do dicionário
df['introducao_contextualizacao'] = df['introducao_dict'].apply(lambda x: x.get('contextualizacao', ''))
df['introducao_problematica'] = df['introducao_dict'].apply(lambda x: x.get('problematica', ''))
df['introducao_ineditismo'] = df['introducao_dict'].apply(lambda x: x.get('ineditismo', ''))
df['introducao_contribuicao'] = df['introducao_dict'].apply(lambda x: x.get('contribuição', ''))

# Remova a coluna temporária
df.drop(columns=['introducao_dict'], inplace=True)

df.head()

,modalidade,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,...,introducao,conclusao,embedding,x,y,z,introducao_contextualizacao,introducao_problematica,introducao_ineditismo,introducao_contribuicao
id,,,,,,,,,,,,,,,,,,,,,
0,treino,Jorge Ivan Hmeljevski,Modelo para Sistemas de Supervisão de Mercado ...,tese,Engenharia do Conhecimento,2021,Florianópolis,"Prof. José Leomar Todesco, Dr.","Prof. Alexandre Leopoldo Gonçalves, Dr.",A confiança na higidez dos mercados de capitai...,...,{'contextualizacao': 'O mercado de capitais é ...,O modelo elaborado nest a pesquisa envolveu a ...,"[0.02262425608932972, 0.06360490620136261, 0.0...",-0.174124,0.159395,-0.060768,O mercado de capitais é fundamental para o cre...,"Os SSM, de maneira geral, usam os dado s prove...","De maneira inédita, portanto, este trabalho pa...","A contribuição desta pesquisa, portanto, está ..."
1,treino,IVAM GALVÃO FILHO,FRACTUS: APLICATIVO PARA APRENDIZAGEM DE FRAÇÕES,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Profª. Vania Ribas Ulbricht, Drª.","Profª. Elisa Maria Pivetta, Drª.",O objetivo principal desta pesquisa foi o dese...,...,{'contextualizacao': 'O sistema educacional br...,O trabalho realizado pela organização TODOS PE...,"[-0.006245349999517202, 0.065089151263237, 0.0...",0.143998,0.263873,-0.055491,O sistema educacional brasileiro passa por uma...,A deficiência na aprendizagem nas escolas de e...,,
2,treino,MÁRCIO CRESCENCIO,MODELO DE UMA REDE COLABORATIVA SUPORTADA POR ...,tese,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Alexandre Augusto Biz , Dr.","Prof. José Leomar Todesco , Dr.",A convergência entre o turismo e a cultura atr...,...,{'contextualizacao': 'O turismo se tornou uma ...,Esta tese identificou que o turismo possui um ...,"[0.024945586919784546, 0.04501194879412651, 0....",-0.111669,-0.167909,-0.015695,O turismo se tornou uma das maiores indústrias...,"A convergência entre o turismo e a cultura, at...",O reconhecimento de PM atrai turistas adiciona...,Esses elementos de ligação do desenvolvimento ...
3,treino,Roseli Honorio,MODELO CONCEITUAL DE GOVERNANÇA DE DADOS COMO ...,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Prof. João Artur de Souza, Dr.","Profa. Patrícia de Sá Freire, Dra.",A humanidade passou por transformações e revol...,...,{'contextualizacao': 'A sociedade está atraves...,"Na Ind ústria 4.0 e Sociedade 5.0, as organiza...","[0.05032069981098175, 0.051356106996536255, 0....",-0.229669,0.045873,-0.059402,A sociedade está atravessando um momento de mu...,É nesse contexto que a Governanç a do Conhecim...,,
4,treino,José Tadeu Silva,Análise da contribuição da engenharia do conhe...,dissertação,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Fernando A. Ostuni Gauthier, Dr.","Prof. Marcelo Macedo, Dr.",A presente dissertação aborda as questões emer...,...,{'contextualizacao': 'De acordo com Lemos (200...,A partir do objetivo geral de analisar as cont...,"[-0.0012473083334043622, 0.044789645820856094,...",-0.195246,0.141014,0.098894,"De acordo com Lemos (2003), sob qualquer aspec...","Para Vilaça e Araújo (2016), o conhecimento so...",,


In [17]:
# Criar o df apenas com os dados que tenham a modalidade treino
df_test = df[df['modalidade'] == 'teste']
df_test

,modalidade,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,...,introducao,conclusao,embedding,x,y,z,introducao_contextualizacao,introducao_problematica,introducao_ineditismo,introducao_contribuicao
id,,,,,,,,,,,,,,,,,,,,,
29,teste,CLÁUDIO DE LIMA,REDES COLABORATIVAS COMO DINÂMICA DE INTERNACI...,tese,Engenharia do Conhecimento,2023,Florianópolis,"Prof. Rogério Cid Bastos , Dr.","Prof. Gregório J. Varvakis Rados, Dr.",As instituições de educação superior precisam ...,...,{'contextualizacao': 'As universidades têm bus...,Esta tese desenvolveu um Modelo de Avaliação d...,"[0.03478163480758667, 0.04431023448705673, 0.0...",-0.054975,0.137685,0.038244,As universidades têm buscado diferentes inicia...,"Por es ses motivos, Palacios -Callender e Robe...",Cabe destacar que a coautoria é considerada um...,"Com base no presente problema de pesquisa, sin..."
30,teste,Rafael Maia Pinto,Detecção de erros e fraudes em gastos públicos...,dissertação,Engenharia do Conhecimento,2023,Florianópolis,"Profa. Lia Caetano Bastos, Dra.","Prof. Rogério Cid Bastos, Dr.",A Constituição Federal estabelece limites míni...,...,{'contextualizacao': 'A Emenda Constitucional ...,À medida que a quantidade e complexidade das t...,"[-0.03038318268954754, 0.02694006636738777, 0....",-0.052560,0.204372,-0.242773,"A Emenda Constitucional 29 (BRASIL, 2000) à Co...","Todos os anos, são gerados milhões de lançamen...",,
31,teste,Elpídio Ribeiro Neves,Sistema sociotécnico de Integração de Modelos ...,tese,Engenharia do Conhecimento,2022,Florianópolis,"Prof. Paulo Maurício Selig, Dr.","Prof. Neri dos Santos, Dr.","As universidades, como organizações complexas,...",...,{'contextualizacao': 'Uma organização é mais q...,"Ao final deste trabalho, apresentam -se as con...","[-0.005148055963218212, 0.045625243335962296, ...",-0.077304,0.032610,-0.028912,Uma organização é mais que um conjunto de bens...,"Em particular, em relaç ão às universidades, c...",Os estudos realizados demonstram que o que ex...,"Este trabalho propõe um artefato, o Sistema SA..."
32,teste,Fernando Ferreira Aguiar,Framework para Criação e Manutenção de Bases d...,tese,Engenharia do Conhecimento,2023,Florianópolis,Prof. Dr. Marcelo Macedo,Prof. Dr. Denilson Sell,A presente tese aborda a problemática das base...,...,{'contextualizacao': 'A inovação impulsiona o ...,"A presente tese, portanto, contribuiu para o c...","[-0.018457775935530663, 0.0481286495923996, 0....",-0.175368,0.130818,0.090165,"A inovação impulsiona o desenvolvimento, trans...",No âmbito da escalabilidade operacional dentro...,,
33,teste,Rudger Nowasky do Nascimento Taxweiler,Framework de Ciência de Dados e Engenharia d...,tese,Engenharia do Conhecimento,2023,Florianópolis,"Denilson Sell, Dr.","Roberto Carlos dos Santos Pacheco, Dr.",A ciência brasileira está fortemente ligada à ...,...,{'contextualizacao': 'A ciência brasileira est...,"O framework ACDK, com base na teoria do Capit...","[0.0138176828622818, 0.06415273994207382, 0.01...",-0.100820,0.090234,0.066427,A ciência brasileira está fortemente ligada à ...,Explorar as informações dos egressos não é alg...,"Na literatura, é possível encontrar diversos t...","Além disso, a própria adoção da metodologia de..."
34,teste,Ricardo Alexandre Diogo,Modelo Conceitual para Formulação de Diretrize...,tese,Engenharia do Conhecimento,2023,Florianópolis,"Prof. Neri dos Santos, Dr.","Prof. Eduardo de Freitas Rocha Loures, Dr.",O fenômeno da Transformação Digital tem trazid...,...,{'contextualizacao': 'A Transformação Digital ...,"O objetivo geral desta tese, de propor um Mode...","[0.02255619876086712, 0.04089098423719406, -0....",-0.025043,0.121479,0.178046,A Transformação Digital da educação tem como p...,A Educação em Engenharia refere-se ao processo...,Apesar de haver na literatura uma proposta de ...,A necessidade de elicitar o conhecimento nos p...
35,teste,Rangel Machado Simon,Educação digital superior: desenvolvendo as co...,tese,Engenharia do 

In [18]:
#Performs queries to group returned concentrarion area for each input dissertation or thesis
print("Index name: ",index_name)
k_list = [1,2,3]
n_list = [1,5,10,15,20]
accuracy_dict = {}
candidate = 105 # 50 e 100
ctr_hit = 0
hits_count = 0
positive = negative = 0
examiner_list = []
hit_list = []
ctr_queries = 0

for index, row in df_test.iterrows():
    ctr_queries+=1
    
    # Converta a string de embedding para uma lista de floats
    embedding = ast.literal_eval(row.embedding)

    hits = search(embedding)

    hit_list.clear()
    hits_count=0
    for hit in hits:
        hit_list.append(hit.payload['area_de_concentracao'])
        hits_count+=1

    print("Query id: "+str(ctr_queries)+" - area_de_concentracao: "+ row.area_de_concentracao  +" - Hits: "+str(hits_count))

    for k in k_list:
        for n in n_list:
            histogram_res = histogram(hit_list[:n], k)
            if (row.area_de_concentracao in histogram_res):
                process_result(accuracy_dict, k, n, 'positive')
            else:
                process_result(accuracy_dict, k, n, 'negative')

print_process_result(accuracy_dict, k_list, n_list)
print("Accuracy by k and n")
matrix = transform_process_result(accuracy_dict, k_list, n_list)
print(matrix)

Index name:  ppgegc_qa_method
Query id: 1 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 2 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 3 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 4 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 5 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 6 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 7 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 8 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 9 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 10 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 11 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 12 - area_de_concentracao: Engenharia do Conhecimento - Hits: 105
Query id: 13 - area_de_concentracao: Engenharia do Conhecimento - H